# HyDRA v7 — GCM Optimal Architecture
**Implementa:** Gemini spec Q1-Q8 + correções do paper GCM (Reis 2026)

Mudanças vs v6:
- ✅ 3-Phase CE reduction **REMOVIDO** (misimplementation do paper)
- ✅ Kuramoto **composite gating** (rdc + logit_std)
- ✅ PCGrad **assimétrico** (CE sempre protegido)
- ✅ **L_align** (distance matrix alignment, GCM Eq.22)
- ✅ HAKORNLayer **sparse** {4,8,11} (pacemaker architecture)
- ✅ VolumeWeightedCE **DESATIVADO** (conflito com SublatticeLMHead)
- ✅ LR 3e-4, warmup 4000, batch 8


Cell 1 — Environment Setup
==========================
Installs dependencies, optionally mounts Google Drive, creates the output
directory tree, and sets global determinism seeds.

Storage strategy (automatic):
  • Google Drive available  → PAPER_ROOT = /content/drive/MyDrive/HydraPaper
  • Drive unavailable/skipped → PAPER_ROOT = /content/HydraPaper
    (persists for the Colab session; download artifacts via Cell 12 ZIP export)

Directory structure:
    /checkpoints/   model snapshots (.pt)
    /logs/          raw training logs (.csv)
    /results/       derived metrics
    /figures/       publication figures (PNG 300 DPI + PDF vector)
    /tables/        LaTeX tables (booktabs format)
    /configs/       JSON reproducibility manifests
    /exports/       final ZIP archive

In [ ]:
import subprocess, sys, os

def _run(cmd):
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if r.returncode != 0:
        print(f'[WARN] {r.stderr[:200]}')
    return r.returncode

# ── numpy fix ANTES de qualquer import ───────────────────────────────────
subprocess.run('pip install -q "numpy==1.26.4" 2>/dev/null', shell=True)

import numpy as _np_check
if tuple(int(x) for x in _np_check.__version__.split('.')[:2]) >= (2, 0):
    print("⚠️  numpy 2.x em memória — reiniciando kernel...")
    import IPython
    IPython.Application.instance().kernel.do_shutdown(restart=True)

print(f'  numpy: {_np_check.__version__} ✅')

# ── H100 dependencies ─────────────────────────────────────────────────────
print('Installing dependencies...')
_run('pip install -q torch tqdm transformers accelerate datasets POT ripser')
_run('pip install -q giotto-tda 2>/dev/null || true')
print('✅ Dependencies installed')

# ── H100 env flags ───────────────────────────────────────────────────────
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
os.environ["TOKENIZERS_PARALLELISM"]  = "false"
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"

# ── Google Drive ──────────────────────────────────────────────────────────
from pathlib import Path
USE_DRIVE = True
try:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
    PAPER_ROOT = Path('/content/drive/MyDrive/HydraPaper_v6').resolve()
    USE_DRIVE  = True
    print('✅ Google Drive mounted')
except Exception as _e:
    PAPER_ROOT = Path('/content/HydraPaper_v6')
    print(f'⚠️  Drive unavailable — local: {PAPER_ROOT}')

for d in ['checkpoints','logs','results','figures','tables','configs','exports']:
    (PAPER_ROOT / d).mkdir(parents=True, exist_ok=True)
print(f'✅ Output root: {PAPER_ROOT}  (drive={USE_DRIVE})')

# ── Determinism ───────────────────────────────────────────────────────────
import torch, random, numpy as np
SEED = 42
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
random.seed(SEED)
torch.backends.cudnn.deterministic    = True
torch.backends.cudnn.benchmark        = False  # False para determinismo
torch.backends.cuda.matmul.allow_tf32 = True   # H100: TF32 OK fora do Lorentz
torch.backends.cudnn.allow_tf32       = True

# numpy seed via nova API
try:
    import numpy.random as _npr; _npr.default_rng(SEED)
except: pass

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

# H100 info
if torch.cuda.is_available():
    gpu_name  = torch.cuda.get_device_name(0)
    gpu_mem   = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'✅ GPU: {gpu_name}  {gpu_mem:.0f}GB VRAM')

print(f'✅ Seeds set  device={DEVICE}  torch={torch.__version__}  numpy={np.__version__}')
import os; os.chdir(PAPER_ROOT)
print(f'📍 Working dir: {Path.cwd()}')


Installing dependencies...
✅ Dependencies installed
⚠️  Drive unavailable (MessageError: Error: credential propagation was unsuccessful)
   → Using local storage: /content/HydraPaper_VariantF
   Artifacts will be in the session ZIP export (Cell 12).
✅ Output root: /content/HydraPaper_VariantF  (drive=True)
✅ Seeds set (seed=42)  device=cuda  torch=2.10.0+cu128
📂 PAPER_ROOT absoluto: /content/HydraPaper_VariantF
📍 Working dir: /content/HydraPaper_VariantF


Cell 2 — Source Upload & Mount  [HARDENED: cache-purge + version lock]
=======================================================================
Purges any cached cgt installation, extracts the uploaded zip fresh,
then PROVES which code version is active before allowing execution to
continue.  If target_radius != 1.5, execution STOPS here.

In [ ]:
import importlib, sys, os, zipfile

# ── STEP 0: Hard purge of all cached cgt modules ───────────
# Colab may hold stale compiled .pyc or already-imported modules in memory.
# Remove them ALL before touching the file system.
_stale = [k for k in sys.modules if k == 'cgt' or k.startswith('cgt.')]
for _k in _stale:
    del sys.modules[_k]
print(f"Purged {len(_stale)} cached cgt module(s) from sys.modules")

# ── STEP 1: Wipe previous extraction on disk ───────────────
import shutil
for _old in ['/content/src', '/content/cgt']:
    if os.path.exists(_old):
        shutil.rmtree(_old)
        print(f"Deleted old directory: {_old}")

# ── STEP 2: Upload and extract ─────────────────────────────
from google.colab import files as _colab_files
print('Upload src_FULL_V5.zip when the dialog appears...')
_uploaded = _colab_files.upload(target_dir='/tmp')
if not _uploaded:
    raise RuntimeError('No file uploaded.')
_zip_name = list(_uploaded.keys())[0]
_zip_path = f'/{_zip_name}'

os.makedirs('/content/src', exist_ok=True)
with zipfile.ZipFile(_zip_path, 'r') as _zf:
    _zf.extractall('/content/src')
print(f'Extracted: {_zip_name}')

# ── STEP 3: Mount sys.path ─────────────────────────────────
_candidates = ['/content/src', '/content/src/src', '/content']
_cgt_root = next(
    (p for p in _candidates if os.path.isdir(os.path.join(p, 'cgt'))), None
)
if _cgt_root is None:
    raise RuntimeError('cgt/ not found in extracted zip. Wrong zip uploaded?')
if _cgt_root not in sys.path:
    sys.path.insert(0, _cgt_root)

# ── STEP 4: Grep the actual file on disk (mandatory) ───────
import subprocess
_distill_path = os.path.join(_cgt_root, 'cgt', 'distillation', 'distillation_v2.py')
if not os.path.exists(_distill_path):
    raise FileNotFoundError(f'distillation_v2.py not found at {_distill_path}')

_grep = subprocess.run(
    ['grep', '-n', 'target_radius', _distill_path],
    capture_output=True, text=True
)
print("\n[DISK VERIFY] grep target_radius distillation_v2.py:")
print(_grep.stdout.strip())

# ── STEP 5: Parse the exact value from file ────────────────
_target_on_disk = None
for _line in _grep.stdout.splitlines():
    if '= ' in _line and 'target_radius' in _line and '#' not in _line.split('target_radius')[0]:
        try:
            _target_on_disk = float(_line.split('=')[1].split('#')[0].strip())
            break
        except ValueError:
            pass

print(f"[DISK VERIFY] target_radius on disk = {_target_on_disk}")

# ── STEP 6: FAIL-FAST if wrong version ─────────────────────
if _target_on_disk is None:
    raise RuntimeError(
        "ABORT: Could not parse target_radius from distillation_v2.py. "
        "Inspect the file manually."
    )
if _target_on_disk >= 2.0:
    raise RuntimeError(
        f"ABORT: target_radius={_target_on_disk} on disk — OLD code loaded. "
        f"Upload src_FULL_V4.zip (not src__7_.zip or any original version). "
        f"Expected: target_radius=1.5"
    )

print(f"\n✅ DISK VERSION CONFIRMED: target_radius={_target_on_disk}")

# ── STEP 7: Import and runtime confirmation ────────────────
import cgt
from cgt.distillation.distillation_v2 import TeacherDistillationLossV2
import inspect, re

_src = inspect.getsource(TeacherDistillationLossV2._radius_loss)
_m = re.search(r'target_radius\s*=\s*([\d.]+)', _src)
_runtime_val = float(_m.group(1)) if _m else None

print(f"[RUNTIME VERIFY] target_radius in loaded class = {_runtime_val}")

if _runtime_val is None or _runtime_val >= 2.0:
    raise RuntimeError(
        f"ABORT: Runtime target_radius={_runtime_val} — module reload failed. "
        f"Kernel may be caching a stale version. "
        f"Go to Runtime → Restart runtime, then re-run from Cell 1."
    )

print(f"✅ RUNTIME VERSION CONFIRMED: target_radius={_runtime_val}")
print(f"✅ cgt mounted  root={_cgt_root}  version={cgt.__version__}")

# ── STEP 8: Verify V4 new modules are present ──────────────
_v4_checks = {
    'HyperbolicProjectorV3': 'cgt.distillation.hyperbolic_projector',
    'GeodesicLMHeadV2':      'cgt.models.geodesic_lm_head',
    'RiemannianAdamW':       'cgt.dynamics.riemannian_adamw',
}
_v4_missing = []
for _cls, _mod in _v4_checks.items():
    try:
        _m = importlib.import_module(_mod)
        getattr(_m, _cls)
        print(f"  ✅ {_cls} present in {_mod}")
    except Exception as _e:
        _v4_missing.append(f"{_cls}: {_e}")
        print(f"  ⚠️  {_cls} missing — {_e}")

if _v4_missing:
    print(f"⚠️  V4 modules missing (non-critical, training still works): {_v4_missing}")
else:
    print("✅ src_FULL_V4.zip confirmed — all new modules present")
print("\n" + "="*60)
print("V4 VERIFIED: target_radius=1.5 + new modules ACTIVE. Safe to proceed.")
print("="*60)

Purged 0 cached cgt module(s) from sys.modules
Deleted old directory: /content/src
Upload src_FULL_V5.zip when the dialog appears...


Saving src.zip to /tmp/src (1).zip
Extracted: /tmp/src (1).zip

[DISK VERIFY] grep target_radius distillation_v2.py:
942:        # FIX: target_radius 4.0 → 1.5
961:        target_radius = 1.5
962:        return F.mse_loss(radius.clamp(max=4.0), torch.full_like(radius, target_radius))
[DISK VERIFY] target_radius on disk = 1.5

✅ DISK VERSION CONFIRMED: target_radius=1.5
[RUNTIME VERIFY] target_radius in loaded class = 1.5
✅ RUNTIME VERSION CONFIRMED: target_radius=1.5
✅ cgt mounted  root=/content/src/src  version=1.0.0
  ✅ HyperbolicProjectorV3 present in cgt.distillation.hyperbolic_projector
  ✅ GeodesicLMHeadV2 present in cgt.models.geodesic_lm_head
  ✅ RiemannianAdamW present in cgt.dynamics.riemannian_adamw
✅ src_FULL_V4.zip confirmed — all new modules present

V4 VERIFIED: target_radius=1.5 + new modules ACTIVE. Safe to proceed.


Cell 3 — HyDRA-Competitive Configuration

In [ ]:
import yaml
from pathlib import Path

PAPER_ROOT = Path("/content/drive/MyDrive/HydraPaper_v7")
SEQ_LEN    = 512

CFG = {
    "model": {
        "vocab_size":        50257,
        "n_embd":            256,   # stable config — float64 attention fits in 80GB
        "n_layer":           12,
        "n_head":            8,
        "n_positions":       SEQ_LEN,
        "hyperbolic_dim":    256,
        "initial_curvature": 1.0,
        "use_dynamics":      True,
    },
    "dynamics": {
        "coupling_strength": 0.3,   # K₀ — initial, will be gated dynamically
        "dt":                0.05,  # max allowed by precision guard
        "num_steps":         1,
    },
    "training": {
        "alpha":             1.0,    # ✅ CE = 1.0 ALWAYS — never reduced
        "temperature":       1.0,
        "lambda_distill":    0.01,   # KL from teacher
        "lambda_hidden":     0.05,   # hidden alignment
        "lambda_radius":     0.05,
        "lambda_contrast":   0.05,
        "label_smoothing":   0.1,
        "learning_rate":     3e-4,   # ✅ Gemini: reduced from 5e-4
        "weight_decay":      0.1,
        "max_steps":         200_000,
        "warmup_steps":      4000,   # ✅ Gemini: extended for CGT softening
        "gradient_clip":     1.0,
        "eval_every":        1000,
        "log_every":         100,
        "checkpoint_every":  1000,
        "lr_floor":          0.05,
        "teacher_hidden_dim":768,
        "batch_size":        8,      # ✅ OOM fix: float64 attention
        "dataset":           "openwebtext",
    },
    "cgt": {
        "enabled":         True,
        "teacher_dim":     768,
        "student_dim":     32,
        "hidden_dim":      256,
        "checkpoint":      None,
        "freeze_init":     False,
        "lambda_cgt":      0.01,
    },
    "topo": {
        "enabled":       True,
        "lambda_topo":   0.10,
        "betti_target":  [1, 0],
        "update_every":  500,
        "max_points":    256,
        "k_neighbors":   10,
        "warmup_steps":  2000,    # ✅ Option A: activates midway through CGT warmup
        "lambda_fr":     0.005,
    },
    "landscape": {
        "enabled":        True,
        "lambda_land":    0.05,
        "n_filtrations":  24,
        "temperature":    0.15,
        "h1_weight":      0.5,
        "update_every":   500,
        "max_points":     128,
        "warmup_steps":   2000,   # ✅ same as topo
    },
    "l_align": {
        # L_align = Σᵢⱼ (dE(eᵢ,eⱼ) - dL(hᵢ,hⱼ))² — GCM paper Eq.22
        "enabled":     True,
        "lambda_align":0.05,     # Gemini recommendation
        "sample_k":    64,       # sampled pairs per step
        "start_step":  100,      # skip cold start
        "every":       50,       # run every N steps
    },
    "gw_monitor": {
        "enabled":        True,
        "update_every":   2000,
        "max_points":     64,
    },
    "binding": {
        "coupling_strength":  0.3,
        "decay_scale":        1.0,
        "frequency_spread":   0.1,
        "dt":                 0.05,
        "integration_method": "rk2",
        "lambda_binding":     0.02,
        "coherence_threshold":0.3,
    },
    "kuramoto_gate": {
        # FrictionAwareCouplingSchedulerV7 — composite gating (Q1)
        "coupling_min":      0.05,
        "coupling_max":      0.30,
        "rdc_danger":        2.0,
        "logit_std_target":  2.5,
    },
    "hpc_v7": {
        # HPCTrainingGuardV7 — CGT softening timeline (Q3)
        "abort_rdc_threshold": 2.0,
        "logit_std_ceiling":   3.0,  # abort if breached before step 4000
        "velocity_limit":      0.5,  # abort if +0.5 per 100 steps
    },
    "early_stopping": {
        "patience":          20,
        "min_delta":         0.005,
        "ema_beta":          0.9,
        "ema_beta_fast":     0.3,
        "min_steps":         20000,
        "window_size":       5,
        "noise_mult":        2.0,
        "plateau_threshold": 10,
        "logit_std_delta":   0.1,
    },
}

print("HyDRA v7 config:")
print(f"  model      : {CFG['model']['n_layer']}L × {CFG['model']['n_embd']}d × {CFG['model']['n_head']}H")
print(f"  seq_len    : {SEQ_LEN}  batch={CFG['training']['batch_size']}")
print(f"  LR         : {CFG['training']['learning_rate']}  warmup={CFG['training']['warmup_steps']} steps")
print(f"  λ_CE={CFG['training']['alpha']}  λ_KL={CFG['training']['lambda_distill']}  λ_hid={CFG['training']['lambda_hidden']}")
print(f"  L_align    : λ={CFG['l_align']['lambda_align']}  sample_k={CFG['l_align']['sample_k']}")
print(f"  topo warmup: step {CFG['topo']['warmup_steps']} (Option A)")
print(f"  KuramotoGate: K∈[{CFG['kuramoto_gate']['coupling_min']}, {CFG['kuramoto_gate']['coupling_max']}]  rdc_danger={CFG['kuramoto_gate']['rdc_danger']}")
print(f"  Pacemakers : layers {{4, 8, 11}}")
print(f"  VolumeWeightedCE: DISABLED")
print(f"  3-Phase CE reduction: REMOVED")
print("✅ CFG v7 ready")

import torch
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"DEVICE: {DEVICE}")
if torch.cuda.is_available():
    total = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"  GPU: {torch.cuda.get_device_name(0)}  {total:.1f}GB")


HyDRA-Competitive config:
  model      : 12L × 512d × 8H
  seq_len    : 256
  max_steps  : 200,000
  batch_size : 4 × 256 = 1,024 tokens/step
  total tokens: 0.2B
  λ_CE=1.0  λ_KL=0.01  λ_hid=0.05
✅ CFG ready
DEVICE: cuda
  GPU: NVIDIA RTX PRO 6000 Blackwell Server Edition  102.0GB
  ✅ sufficient memory for 12L×512d


Cell 5 — Teacher + Student

In [ ]:
import time
from transformers import GPT2Tokenizer
from cgt.api.entrypoint import SafeHyperbolicModel, SafeModelConfig
from cgt.distillation import GPT2TeacherWrapperV2
from cgt.models.geodesic_lm_head import AngularLMHeadV2
from cgt.models.sublattice_lm_head import SublatticeLMHead

# ── Tokenizer ─────────────────────────────────────────────────────────────
_tok_cache = PAPER_ROOT / "models" / "gpt2_cache"
if _tok_cache.exists():
    tokenizer = GPT2Tokenizer.from_pretrained(str(_tok_cache))
    print(f"Tokenizer: loaded from Drive cache")
else:
    tokenizer = GPT2Tokenizer.from_pretrained("gpt2")
    tokenizer.save_pretrained(str(_tok_cache))
    print(f"Tokenizer: downloaded + cached to Drive")
tokenizer.pad_token = tokenizer.eos_token
VOCAB_SIZE = tokenizer.vocab_size
print(f"Tokenizer: vocab={VOCAB_SIZE}")

# ── Teacher ───────────────────────────────────────────────────────────────
_model_drive = PAPER_ROOT / "models" / "gpt2_model_cache"
if _model_drive.exists() and any(_model_drive.iterdir()):
    print("Teacher: loading from Drive cache...")
    teacher = GPT2TeacherWrapperV2(model_name=str(_model_drive), device=DEVICE)
    print(f"Teacher: GPT-2-small {teacher.config.n_embd}d [frozen] — from cache")
else:
    print("Teacher: downloading GPT-2-small (~548MB)...")
    teacher = GPT2TeacherWrapperV2(model_name="gpt2", device=DEVICE)
    _model_drive.mkdir(parents=True, exist_ok=True)
    teacher.model.save_pretrained(str(_model_drive))
    print(f"Teacher: GPT-2-small {teacher.config.n_embd}d [frozen] — saved to Drive")

# ── Student ───────────────────────────────────────────────────────────────
torch.manual_seed(42)
student_cfg = SafeModelConfig(
    vocab_size        = VOCAB_SIZE,
    n_embd            = CFG["model"]["n_embd"],
    n_layer           = CFG["model"]["n_layer"],
    n_head            = CFG["model"]["n_head"],
    n_positions       = CFG["model"]["n_positions"],
    initial_curvature = CFG["model"]["initial_curvature"],
    use_dynamics      = CFG["model"]["use_dynamics"],
    hyperbolic_dim    = CFG["model"]["hyperbolic_dim"],
    coupling_strength = CFG["dynamics"]["coupling_strength"],
    dt                = CFG["dynamics"]["dt"],
    num_steps         = CFG["dynamics"]["num_steps"],
)
student = SafeHyperbolicModel(student_cfg).to(DEVICE)

# ── SublatticeLMHead (replaces default head — DegEq-immune + Zipf hierarchy) ─
try:
    _substrate = student.core_model.encoder.layers[0].attention.substrate
    _sublattice_head = SublatticeLMHead(
        n_embd             = CFG["model"]["n_embd"],
        vocab_size         = CFG["model"]["vocab_size"],
        substrate          = _substrate,
        r_max_frequent     = 1.5,    # frequent tokens near origin
        r_max_rare         = 5.0,    # rare tokens near boundary (Zipf)
        frequency_threshold= 0.3,
        learnable_scale    = True,
        angular_mode       = True,   # ∂logit/∂r = 0 — Channel 2 DegEq fix
    )
    student.core_model.lm_head = _sublattice_head
    student = student.to(DEVICE)     # remount after head swap — fixes CPU/CUDA mismatch
    print(f"  lm_head: SublatticeLMHead installed ✅")
    print(f"    Sublattice A (frequent): r_max=1.5  |  Sublattice B (rare): r_max=5.0")
    print(f"    angular_mode=True → ∂logit/∂r = 0 (DegEq immune)")
except Exception as _sl_e:
    print(f"  ⚠️  SublatticeLMHead failed: {_sl_e}")
    print(f"  Falling back to AngularLMHeadV2...")
    _sub = student.core_model.substrate
    student.core_model.lm_head = AngularLMHeadV2(
        n_embd       = CFG["model"]["n_embd"],
        vocab_size   = VOCAB_SIZE,
        substrate    = _sub,
        temperature  = 20.0,
        learnable_temp = False,
    ).to(DEVICE)
    print(f"  lm_head: AngularLMHeadV2 (fallback)")

n_params = student.num_parameters()
_head_name = type(student.core_model.lm_head).__name__
print(f"\nStudent: {n_params:,} params ({n_params/1e6:.1f}M)")
print(f"  arch    : {student_cfg.n_layer}L × {student_cfg.n_embd}d × {student_cfg.n_head}H")
print(f"  dynamics: use_dynamics={student_cfg.use_dynamics}  K={student_cfg.coupling_strength}  dt={student_cfg.dt}")
print(f"  lm_head : {_head_name}")
print(f"  teacher/student ratio: {117e6/n_params:.1f}x")
print("✅ Models ready")
# ── HyDRA v3: H-AKORN Temporal Binding ──────────────────────────────────
# Replaces simple Kuramoto with geometry-aware phase coupling:
# dθᵢ/dt = ωᵢ + Σⱼ K₀·exp(-d_H(i,j)/τ)·sin(θⱼ−θᵢ)
from cgt.psi_extensions.binding import HAKORNLayer, PhaseCoherenceLoss
from cgt.psi_extensions.binding.h_akorn import HAKORNConfig

try:
    _substrate = student.core_model.encoder.layers[0].attention.substrate
    _hakorn_cfg = HAKORNConfig(
        coupling_strength  = CFG["binding"]["coupling_strength"],
        decay_scale        = CFG["binding"]["decay_scale"],
        frequency_spread   = CFG["binding"]["frequency_spread"],
        dt                 = CFG["binding"]["dt"],
        integration_method = CFG["binding"]["integration_method"],
    )
    # Attach one HAKORNLayer per encoder layer
    _seq_len = CFG["model"]["n_positions"]  # 256 — num_nodes per step
    for _layer in student.core_model.encoder.layers:
        _layer.hakorn = HAKORNLayer(
            num_nodes         = _seq_len,
            coupling_strength = CFG["binding"]["coupling_strength"],
            temperature       = CFG["binding"]["decay_scale"],
            dt                = CFG["binding"]["dt"],
        ).to(DEVICE)
    # Phase coherence loss (used in Cell 13)
    phase_coherence_loss = PhaseCoherenceLoss(
        critical_threshold = CFG["binding"]["coherence_threshold"]
    ).to(DEVICE)
    student = student.to(DEVICE)
    print(f"  [H-AKORN] ✅ Temporal binding active")
    print(f"    K₀={CFG['binding']['coupling_strength']}  τ={CFG['binding']['decay_scale']}  dt={CFG['binding']['dt']}")
    print(f"    dθ/dt = ω + Σⱼ K₀·exp(-d_H/τ)·sin(θⱼ−θᵢ)")
except Exception as _hae:
    phase_coherence_loss = None
    print(f"  [H-AKORN] ⚠️  {_hae} — falling back to standard Kuramoto")
# ── HyDRA v4: CGT Initialization of SublatticeLMHead ─────────────────────
# CGT compresses GPT-2 token embeddings (50257×768) into H^n (50257×32)
# guaranteeing F1 (Lorentz), F2 (distance), F3 (neighborhood) from the paper.
# These structured hyperbolic positions initialize weight_frequent + weight_rare
# so tokens start semantically placed before training — accelerating Kuramoto sync.
if CFG["cgt"]["enabled"]:
    try:
        from cgt.models.cgt_hardened import CGTStudentHardened
        from cgt.geometry.lorentz_v2 import LorentzSubstrateV2

        # ── Build CGT student ─────────────────────────────────────────────
        _cgt_student = CGTStudentHardened(
            teacher_dim = CFG["cgt"]["teacher_dim"],
            student_dim = CFG["cgt"]["student_dim"],
            hidden_dim  = CFG["cgt"]["hidden_dim"],
        ).to(DEVICE)

        # Load trained CGT weights if available
        _cgt_ckpt = CFG["cgt"]["checkpoint"]
        if _cgt_ckpt and Path(_cgt_ckpt).exists():
            _cgt_student.load_state_dict(torch.load(_cgt_ckpt, map_location=DEVICE))
            _cgt_student.eval()
            print(f"  [CGT] ✅ Loaded checkpoint: {_cgt_ckpt}")
        else:
            print(f"  [CGT] ⚠️  No checkpoint — using random CGT init")

        # ── Encode GPT-2 token embeddings → H^n ──────────────────────────
        _wte = teacher.model.transformer.wte.weight.detach()  # [V, 768]
        _V   = _wte.shape[0]  # 50257

        # Process in chunks to avoid OOM
        _CHUNK_SIZE = 1000
        _hyp_vecs = []
        with torch.no_grad():
            for _ci in range(0, _V, _CHUNK_SIZE):
                _chunk = _wte[_ci:_ci+_CHUNK_SIZE].float()
                _h = _cgt_student(_chunk)  # [chunk, 32+1] Lorentz
                _hyp_vecs.append(_h.detach().cpu())
        _hyp_all = torch.cat(_hyp_vecs, dim=0)  # [V, 33]
        print(f"  [CGT] ✅ Encoded {_V:,} tokens → H^{CFG['cgt']['student_dim']}  shape={list(_hyp_all.shape)}")

        # ── Project 33 → n_embd+1 to match SublatticeLMHead ─────────────
        _n_target = CFG["model"]["n_embd"] + 1  # 257
        if _hyp_all.shape[1] != _n_target:
            # Linear projection preserving Lorentz structure
            # Time component stays, spatial components projected
            _t_comp = _hyp_all[:, :1]  # [V, 1] time
            _s_comp = _hyp_all[:, 1:]  # [V, 32] spatial
            _proj   = torch.nn.Linear(_s_comp.shape[1], _n_target-1, bias=False)
            torch.nn.init.orthogonal_(_proj.weight)  # orthogonal = isometry-preserving
            _s_proj = _proj(_s_comp.float())  # [V, n_embd]
            _hyp_proj = torch.cat([_t_comp, _s_proj], dim=1)  # [V, n_embd+1]
        else:
            _hyp_proj = _hyp_all

        # ── Initialize SublatticeLMHead from CGT projections ─────────────
        _lm = student.core_model.lm_head
        if hasattr(_lm, "freq_indices") and hasattr(_lm, "rare_indices"):
            with torch.no_grad():
                # Extract spatial components (drop time dim) for weight init
                _spatial = _hyp_proj[:, 1:].to(DEVICE)  # [V, n_embd]
                _freq_w  = _spatial[_lm.freq_indices]   # [n_freq, n_embd]
                _rare_w  = _spatial[_lm.rare_indices]   # [n_rare, n_embd]
                # Copy to SublatticeLMHead weights
                _lm.weight_frequent.data.copy_(_freq_w)
                _lm.weight_rare.data.copy_(_rare_w)
            print(f"  [CGT] ✅ SublatticeLMHead initialized:")
            print(f"    freq weights: {_freq_w.shape} (r≤1.5)")
            print(f"    rare weights: {_rare_w.shape} (r≤5.0)")
            print(f"    Tokens start semantically placed in H^n — F1/F2/F3 guaranteed")
        else:
            print("  [CGT] ⚠️  SublatticeLMHead not found — CGT init skipped")

        # Freeze if requested
        if CFG["cgt"]["freeze_init"]:
            for p in student.core_model.lm_head.parameters():
                p.requires_grad = False
            print(f"  [CGT] 🔒 SublatticeLMHead frozen for first {CFG['training']['warmup_steps']} steps")

        student = student.to(DEVICE)
        print(f"  [CGT] ✅ v4 CGT initialization complete")
        del _cgt_student, _hyp_vecs, _hyp_all  # free memory
        torch.cuda.empty_cache()

    except Exception as _cgt_e:
        import traceback
        print(f"  [CGT] ⚠️  Init failed: {_cgt_e}")
        traceback.print_exc()
        print(f"  [CGT] Continuing with random initialization")
else:
    print("  [CGT] ℹ️  Disabled — using standard SublatticeLMHead init")



Tokenizer: loaded from Drive cache
Tokenizer: vocab=50257
Teacher: loading from Drive cache...


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

Teacher: GPT-2-small 768d [frozen] — from cache
  lm_head: SublatticeLMHead installed ✅
    Sublattice A (frequent): r_max=1.5  |  Sublattice B (rare): r_max=5.0
    angular_mode=True → ∂logit/∂r = 0 (DegEq immune)

Student: 63,563,105 params (63.6M)
  arch    : 12L × 512d × 8H
  dynamics: use_dynamics=True  K=0.3  dt=0.05
  lm_head : SublatticeLMHead
  teacher/student ratio: 1.8x
✅ Models ready
  [H-AKORN] ✅ Temporal binding active
    K₀=0.3  τ=1.0  dt=0.05
    dθ/dt = ω + Σⱼ K₀·exp(-d_H/τ)·sin(θⱼ−θᵢ)


Cell 6 — OpenWebText DataLoader  [HARDENED: Drive cache → fallback download]
=============================================================================
Estratégia de cache em 3 camadas:
  1. Drive cache (HF Arrow format) → carrega em < 2 min se já baixou antes
  2. /content cache local (sessão ainda viva) → instantâneo
  3. Download fresco → só se as duas camadas acima falharem (~10-30 min)

Variável de controle:
  FORCE_REDOWNLOAD = True   → ignora cache e baixa tudo de novo (use só se corrompeu)
  FORCE_REDOWNLOAD = False  → comportamento normal (padrão)

In [ ]:
import os
from pathlib import Path

FORCE_REDOWNLOAD = False  # ← mude para True só se o cache estiver corrompido

# ── Diretórios de cache ───────────────────────────────────────────────────
_HF_DRIVE_CACHE  = PAPER_ROOT / "hf_cache"          # cache persistente no Drive
_HF_LOCAL_CACHE  = Path("/content/hf_cache_local")  # cache rápido na sessão

_HF_DRIVE_CACHE.mkdir(parents=True, exist_ok=True)
_HF_LOCAL_CACHE.mkdir(parents=True, exist_ok=True)

# Aponta HF para o cache do Drive como primário
os.environ["HF_DATASETS_CACHE"] = str(_HF_DRIVE_CACHE)
os.environ["HF_HOME"]           = str(_HF_DRIVE_CACHE)

if FORCE_REDOWNLOAD:
    os.environ["HF_DATASETS_OFFLINE"] = "0"
    print("⚠️  FORCE_REDOWNLOAD=True — ignorando cache, baixando tudo de novo")
else:
    # Tenta offline primeiro; se falhar, libera download
    os.environ["HF_DATASETS_OFFLINE"] = "1"
    print(f"📦 Cache Drive : {_HF_DRIVE_CACHE}")
    print(f"📦 Cache local : {_HF_LOCAL_CACHE}")

from cgt.distillation.dataset_v2 import build_openwebtext_loaders

def _build_loaders(offline: bool):
    os.environ["HF_DATASETS_OFFLINE"] = "1" if offline else "0"
    return build_openwebtext_loaders(
        tokenizer,
        seq_len           = CFG["model"]["n_positions"],
        batch_size        = CFG["training"]["batch_size"],
        overlap           = 64,
        num_workers       = 2,
        max_train_samples = 500_000,
    )

train_loader = val_loader = None

# Camada 1: tenta carregar do cache (modo offline)
if not FORCE_REDOWNLOAD:
    try:
        print("🔍 Tentando carregar do cache Drive (modo offline)...")
        train_loader, val_loader = _build_loaders(offline=True)
        print("✅ Dataset carregado do cache Drive — sem download!")
    except Exception as _cache_err:
        print(f"⚠️  Cache offline falhou: {type(_cache_err).__name__}: {_cache_err}")
        print("   → Ativando download...")

# Camada 2/3: download se necessário
if train_loader is None:
    try:
        print("🌐 Baixando OpenWebText (~12GB comprimido, 10-30min na primeira vez)...")
        train_loader, val_loader = _build_loaders(offline=False)
        print("✅ Download concluído — cache salvo em Drive para próxima sessão")
    except Exception as _dl_err:
        raise RuntimeError(
            f"Falha ao carregar dataset: {_dl_err}\n"
            f"Verifique conexão e espaço no Drive ({_HF_DRIVE_CACHE})"
        ) from _dl_err

print(f"\nTrain batches : {len(train_loader):,}")
print(f"Val   batches : {len(val_loader):,}")
print(f"Tokens/step   : {CFG['training']['batch_size'] * CFG['model']['n_positions']:,}")
print(f"Cache Drive   : {_HF_DRIVE_CACHE}")
print("✅ Dataset ready")

📦 Cache Drive : /content/drive/MyDrive/HydraPaper_v4/hf_cache
📦 Cache local : /content/hf_cache_local
🔍 Tentando carregar do cache Drive (modo offline)...
  [OpenWebText] Loading dataset...
  [OpenWebText] Cache hit: /content/drive/MyDrive/HydraPaper_VariantF/data/openwebtext


README.md: 0.00B [00:00, ?B/s]

Resolving data files:   0%|          | 0/80 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/80 [00:00<?, ?it/s]

Loading dataset shards:   0%|          | 0/80 [00:00<?, ?it/s]

Token indices sequence length is longer than the specified maximum sequence length for this model (1217 > 1024). Running this sequence through the model will result in indexing errors


  [OpenWebText] Capped to 500,000 samples
  [OpenWebText] 500,000 documents
  [OpenWebText] Tokenising 500,000 docs in-memory...
  [OpenWebText]   50,000/500,000  tokens so far: 56,737,332
  [OpenWebText]   100,000/500,000  tokens so far: 112,963,538
  [OpenWebText]   150,000/500,000  tokens so far: 170,126,390
  [OpenWebText]   200,000/500,000  tokens so far: 226,867,891
  [OpenWebText]   250,000/500,000  tokens so far: 283,174,962
  [OpenWebText]   300,000/500,000  tokens so far: 340,394,624
  [OpenWebText]   350,000/500,000  tokens so far: 397,322,913
  [OpenWebText]   400,000/500,000  tokens so far: 453,973,661
  [OpenWebText]   450,000/500,000  tokens so far: 510,056,246
  [OpenWebText]   500,000/500,000  tokens so far: 566,334,506
  [OpenWebText] 2,802,175 train / 147,482 val windows
✅ Dataset carregado do cache Drive — sem download!

Train batches : 700,544
Val   batches : 36,871
Tokens/step   : 1,024
Cache Drive   : /content/drive/MyDrive/HydraPaper_v4/hf_cache
✅ Dataset ready


In [ ]:
import os
os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"
import torch
torch.cuda.empty_cache()

Cell 7 — Competitive Training

In [ ]:
import time, csv, torch
from pathlib import Path
from cgt.distillation import DistillationConfigV2, DistillationTrainerV2, RadiusCollapseAbort
from cgt.diagnostics.hysteresis  import HysteresisDetector
from cgt.dynamics.pcgrad         import PCGrad
from cgt.dynamics.geometric_controller import GeometricController, ControllerConfig
from cgt.losses.anosov_topo_loss import AnosovTopoLoss, AnosovTopoConfig, FormanRicciRegularizer
from cgt.losses.persistence_landscape import PersistenceLandscapeLoss
from cgt.diagnostics.gw_monitor import GWDivergenceMonitor
# ── v7 modules ────────────────────────────────────────────────────────────
from cgt.distillation.active_integration_v7 import (
    FrictionAwareCouplingSchedulerV7,
    HPCTrainingGuardV7,
    activate_all_modules_v7,
    attach_sparse_hakorn,
    PACEMAKER_LAYERS,
)

RUN_TAG   = "hydra_v7_gcm"
MAX_STEPS = CFG["training"]["max_steps"]

_log_dir  = PAPER_ROOT / "logs"        / RUN_TAG
_ckpt_dir = PAPER_ROOT / "checkpoints" / RUN_TAG
_log_dir.mkdir(parents=True, exist_ok=True)
_ckpt_dir.mkdir(parents=True, exist_ok=True)
print(f"Run → {_log_dir}")

print(f"  Checkpoint dir: {_ckpt_dir}")
ckpts = list(_ckpt_dir.glob("*.pt"))
print(f"  Checkpoints: {[c.name for c in ckpts]}")

# ── Config ────────────────────────────────────────────────────────────────
dist_cfg = DistillationConfigV2(
    alpha                    = CFG["training"]["alpha"],          # 1.0 — NEVER changes
    temperature              = CFG["training"]["temperature"],
    lambda_distill           = CFG["training"]["lambda_distill"],
    lambda_hidden            = CFG["training"]["lambda_hidden"],
    lambda_radius            = CFG["training"]["lambda_radius"],
    lambda_contrast          = CFG["training"]["lambda_contrast"],
    label_smoothing          = CFG["training"]["label_smoothing"],
    learning_rate            = CFG["training"]["learning_rate"],
    weight_decay             = CFG["training"]["weight_decay"],
    max_steps                = MAX_STEPS,
    warmup_steps             = CFG["training"]["warmup_steps"],
    gradient_clip            = CFG["training"]["gradient_clip"],
    eval_every               = CFG["training"]["eval_every"],
    log_every                = CFG["training"]["log_every"],
    checkpoint_every         = CFG["training"]["checkpoint_every"],
    lr_floor                 = CFG["training"]["lr_floor"],
    teacher_hidden_dim       = CFG["training"]["teacher_hidden_dim"],
    riemannian_correct_vocab   = True,
    riemannian_correct_embed   = True,
    riemannian_correct_encoder = True,
    use_oted                 = True,
    oted_r_target            = 1.5,
    use_angular_head         = False,
    adaptive_lambda          = False,
    deg_eq_action            = "none",
    early_stopping_patience  = CFG["early_stopping"]["patience"],
    early_stopping_min_delta = CFG["early_stopping"]["min_delta"],
    ema_beta                 = CFG["early_stopping"]["ema_beta"],
    ema_beta_fast            = CFG["early_stopping"]["ema_beta_fast"],
    min_steps_before_stopping= CFG["early_stopping"]["min_steps"],
    trend_window             = CFG["early_stopping"]["window_size"],
    noise_mult               = CFG["early_stopping"]["noise_mult"],
    plateau_threshold        = CFG["early_stopping"]["plateau_threshold"],
    logit_std_min_delta      = CFG["early_stopping"]["logit_std_delta"],
    keep_last_n_checkpoints  = 3,
    radius_collapse_abort    = True,
)
dist_cfg.deg_eq_action  = "none"
dist_cfg.temperature    = 1.0

# ── Resume check ──────────────────────────────────────────────────────────
_best_ckpt = _ckpt_dir / "distill_v2_best.pt"
_resume    = _best_ckpt.exists()
print(f"  {'🔄 Resuming' if _resume else '🆕 Starting'} — {_best_ckpt.name if _resume else 'from scratch'}")

# ── Memory setup ──────────────────────────────────────────────────────────
import torch as _tc
_tc.cuda.empty_cache()
_tc.set_float32_matmul_precision("high")
_tc.backends.cuda.matmul.allow_tf32 = False  # MANDATORY for Lorentz operations
_tc.backends.cudnn.allow_tf32       = False
print("  ℹ️  Running in eager mode (torch.compile disabled — graph breaks in H-AKORN)")

# ── ✅ v7: Sparse HAKORNLayer attachment {4,8,11} ──────────────────────────
attach_sparse_hakorn(student, CFG, str(DEVICE))

# ── Trainer ───────────────────────────────────────────────────────────────
trainer = DistillationTrainerV2(
    student        = student,
    teacher        = teacher,
    config         = dist_cfg,
    tokenizer      = tokenizer,
    checkpoint_dir = _ckpt_dir,
    device         = DEVICE,
)

if _resume:
    try:
        trainer.load_checkpoint(_best_ckpt)
        print(f"  ✅ Resumed at step {trainer.step}  best_val={trainer.best_val:.4f}")
        dist_cfg.deg_eq_action  = "none"
        dist_cfg.lambda_distill = CFG["training"]["lambda_distill"]
        dist_cfg.lambda_hidden  = CFG["training"]["lambda_hidden"]
        dist_cfg.temperature    = 1.0
    except Exception as e:
        print(f"  ⚠️  Resume failed: {e} — starting fresh")

if not _resume:
    trainer.save(is_best=True)
    import time as _t; _t.sleep(3)
    _ckpts_now = list(_ckpt_dir.glob("*.pt"))
    print(f"  💾 Checkpoint inicial salvo → {[c.name for c in _ckpts_now]}")

# ── AdaptiveHyperController ───────────────────────────────────────────────
try:
    from cgt.distillation import AdaptiveHyperController, AdaptiveControllerConfig
    controller = AdaptiveHyperController(
        config = AdaptiveControllerConfig(
            grad_balance_every = 500,
            balance_tolerance  = 2.0,
            balance_lr         = 0.5,
            ppl_stall_patience = 5,
            lr_reduce_factor   = 0.7,
            regime_every       = 200,
            rdc_danger         = 3.0,
            logit_std_max      = 15.0,
            verbose            = True,
        ),
        trainer = trainer,
    )
    trainer.adaptive_controller = controller
    print(f"  [AdaptiveController] ✅ Activo")
except ImportError:
    class _Stub:
        cfg = type('cfg', (), {'grad_balance_every': 5000, 'regime_every': 200})()
        def summary(self): return "controller não carregado"
    controller = _Stub()

# ── Binding loss hook — only active pacemaker layers ─────────────────────
_lambda_binding = CFG["binding"]["lambda_binding"]

def _binding_loss_hook(trainer_self):
    if phase_coherence_loss is None: return None
    order_params = []
    try:
        for idx, layer in enumerate(trainer_self.student.core_model.encoder.layers):
            if idx in PACEMAKER_LAYERS and hasattr(layer, "hakorn") and layer.hakorn is not None:
                phases = layer.hakorn.phases
                order  = torch.abs(torch.mean(torch.exp(1j * phases.float()))).real
                order_params.append(order)
        if order_params:
            mean_order = torch.stack(order_params).mean()
            return phase_coherence_loss(mean_order) * _lambda_binding
    except Exception: pass
    return None

trainer._extra_loss_hooks = [_binding_loss_hook]
print(f"  [v3] ✅ Binding loss hook: λ={_lambda_binding}  Γ_target={CFG['binding']['coherence_threshold']}")

# ── CGT coherence hook ────────────────────────────────────────────────────
_lambda_cgt = CFG["cgt"]["lambda_cgt"]

def _cgt_coherence_hook(trainer_self):
    try:
        lm = trainer_self.student.core_model.lm_head
        if not (hasattr(lm, "weight_frequent") and hasattr(lm, "_get_substrate")):
            return None
        sub = lm._get_substrate() if hasattr(lm, "_get_substrate") else None
        if sub is None: return None
        w = lm.weight_frequent
        inner = sub.minkowski_inner(w.unsqueeze(-2), w.unsqueeze(-2)).squeeze()
        return (inner + 1.0).abs().mean() * _lambda_cgt
    except Exception: return None

trainer._extra_loss_hooks.append(_cgt_coherence_hook)
print(f"  [v4] ✅ CGT coherence hook: λ={_lambda_cgt}")

# ── Persistence Landscape ─────────────────────────────────────────────────
_land_loss = PersistenceLandscapeLoss(
    lambda_topo   = CFG["landscape"]["lambda_land"],
    n_filtrations = CFG["landscape"]["n_filtrations"],
    temperature   = CFG["landscape"]["temperature"],
    h1_weight     = CFG["landscape"]["h1_weight"],
    warmup_steps  = CFG["landscape"]["warmup_steps"],
    update_every  = CFG["landscape"]["update_every"],
    max_points    = CFG["landscape"]["max_points"],
) if CFG["landscape"]["enabled"] else None

def _landscape_hook(trainer_self):
    if _land_loss is None: return None
    cur_step = getattr(trainer_self, "step", 0)
    if cur_step % _land_loss.update_every != 0: return None
    try:
        lm = trainer_self.student.core_model.lm_head
        W  = lm._get_all_weights() if hasattr(lm, "_get_all_weights") else lm.weight
        spatial = W[:, 1:].float() if W.shape[-1] > _land_loss.max_points else W.float()
        idx  = torch.randperm(spatial.shape[0])[:_land_loss.max_points]
        loss, _ = _land_loss(spatial[idx], step=cur_step)
        return loss if loss.item() > 0 else None
    except Exception: return None

if CFG["landscape"]["enabled"]:
    trainer._extra_loss_hooks.append(_landscape_hook)
    print(f"  [v6] ✅ Persistence Landscape: λ={CFG['landscape']['lambda_land']}  warmup@{CFG['landscape']['warmup_steps']}")

# ── AnosovTopoLoss ────────────────────────────────────────────────────────
if CFG["topo"]["enabled"]:
    _topo_cfg = AnosovTopoConfig(
        lambda_topo  = CFG["topo"]["lambda_topo"],
        betti_target = CFG["topo"]["betti_target"],
        update_every = CFG["topo"]["update_every"],
        max_points   = CFG["topo"]["max_points"],
        k_neighbors  = CFG["topo"]["k_neighbors"],
        warmup_steps = CFG["topo"]["warmup_steps"],   # ✅ step 2000 (Option A)
    )
    _topo_loss = AnosovTopoLoss(_topo_cfg)
    _fr_reg    = FormanRicciRegularizer(
        lambda_fr   = CFG["topo"]["lambda_fr"],
        k_neighbors = CFG["topo"]["k_neighbors"],
    )

    def _topo_hook(trainer_self):
        cur_step = getattr(trainer_self, "step", 0)
        if cur_step % _topo_cfg.update_every != 0: return None
        try:
            lm = trainer_self.student.core_model.lm_head
            spatial = (lm._get_all_weights()[:, 1:] if hasattr(lm, "_get_all_weights")
                       else lm.weight).detach().float()
            topo, _ = _topo_loss(spatial, step=cur_step)
            fr      = _fr_reg(spatial)
            total   = topo + fr
            return total if total.item() > 0 else None
        except Exception: return None

    trainer._extra_loss_hooks.append(_topo_hook)
    print(f"  [v6] ✅ AnosovTopoLoss: λ={CFG['topo']['lambda_topo']}  warmup@{CFG['topo']['warmup_steps']} (Option A)")

# ── GW Monitor ────────────────────────────────────────────────────────────
_gw_monitor = GWDivergenceMonitor(
    update_every = CFG["gw_monitor"]["update_every"],
    max_points   = CFG["gw_monitor"]["max_points"],
) if CFG["gw_monitor"]["enabled"] else None

def _gw_monitor_hook(trainer_self):
    if _gw_monitor is None: return None
    cur_step = getattr(trainer_self, "step", 0)
    try:
        lm    = trainer_self.student.core_model.lm_head
        s_emb = lm._get_all_weights()[:, 1:].detach() if hasattr(lm, "_get_all_weights") else None
        t_emb = trainer_self.teacher.model.transformer.wte.weight.detach()
        if s_emb is None: return None
        N   = min(s_emb.shape[0], t_emb.shape[0], CFG["gw_monitor"]["max_points"])
        idx = torch.randperm(N)
        dgw = _gw_monitor.step(s_emb[idx], t_emb[idx], training_step=cur_step)
        if dgw is not None:
            if _gw_monitor.phase_transition_detected():
                print(f"  [v7/GW] ⚡ PHASE TRANSITION @ step={cur_step}  DGW={dgw:.4f}")
            else:
                print(f"  [v7/GW] {_gw_monitor.report()}")
    except Exception: pass
    return None

if CFG["gw_monitor"]["enabled"]:
    trainer._extra_loss_hooks.append(_gw_monitor_hook)
    print(f"  [v6] ✅ GW Monitor: update@{CFG['gw_monitor']['update_every']}steps")

# ── ✅ v7: HPCTrainingGuardV7 ──────────────────────────────────────────────
hpc_guard_v7 = HPCTrainingGuardV7(
    abort_rdc_threshold = CFG["hpc_v7"]["abort_rdc_threshold"],
    logit_std_ceiling   = CFG["hpc_v7"]["logit_std_ceiling"],
    velocity_limit      = CFG["hpc_v7"]["velocity_limit"],
    warmup_steps        = CFG["training"]["warmup_steps"],
)
print(f"  [v7] ✅ HPCTrainingGuardV7: logit_ceil={CFG['hpc_v7']['logit_std_ceiling']}  vel_limit={CFG['hpc_v7']['velocity_limit']}")

hysteresis_det = HysteresisDetector(window_size=200)
print("  [v7] ✅ HysteresisDetector active")

# ── ✅ v7: PCGrad (asymmetric — CE protected) ──────────────────────────────
try:
    pcgrad_inst = PCGrad(reduction="mean")
    print("  [v7] ✅ PCGrad (asymmetric CE-priority) active")
except Exception as _pe:
    pcgrad_inst = None
    print(f"  [v7] ⚠️  PCGrad: {_pe}")

# ── ✅ v7: FrictionAwareCouplingSchedulerV7 (composite gating) ─────────────
kuramoto_scheduler = FrictionAwareCouplingSchedulerV7(
    coupling_min     = CFG["kuramoto_gate"]["coupling_min"],
    coupling_max     = CFG["kuramoto_gate"]["coupling_max"],
    rdc_danger       = CFG["kuramoto_gate"]["rdc_danger"],
    logit_std_target = CFG["kuramoto_gate"]["logit_std_target"],
)
print(f"  [v7] ✅ KuramotoGate: K∈[{CFG['kuramoto_gate']['coupling_min']},{CFG['kuramoto_gate']['coupling_max']}]  rdc_danger={CFG['kuramoto_gate']['rdc_danger']}")

# ── ✅ v7: VolumeWeightedCE DISABLED ──────────────────────────────────────
vol_ce = None  # DISABLED — conflicts with SublatticeLMHead radial bounds
print("  [v7] ✅ VolumeWeightedCE: DISABLED (SublatticeLMHead conflict)")

# ── GeometricController ───────────────────────────────────────────────────
try:
    geo_ctrl = GeometricController(
        loss_fn=trainer.loss_fn,
        config=ControllerConfig(rdc_safe=2.0, rdc_warning=5.0,
                                rdc_critical=8.0, warmup_steps=500),
    )
    print("  [v7] ✅ GeometricController active")
except Exception as _ge:
    geo_ctrl = None

# ── SGDR ──────────────────────────────────────────────────────────────────
_cur_step = trainer.step
if _cur_step >= 50000:
    for _pg in trainer.optimizer.param_groups:
        _pg["lr"] = 1.5e-4
    print(f"  [SGDR] ✅ Warm restart @ step {_cur_step}")
else:
    print(f"  [SGDR] Starting fresh from step {_cur_step}")

# ── Preflight summary ─────────────────────────────────────────────────────
_head = type(getattr(getattr(trainer.student,"core_model",trainer.student),"lm_head",None)).__name__
_active_pacemakers = [i for i, l in enumerate(trainer.student.core_model.encoder.layers)
                      if hasattr(l, "hakorn") and l.hakorn is not None]
print(f"  lm_head     : {_head}")
print(f"  pacemakers  : {_active_pacemakers}")
print(f"  binding     : H-AKORN sparse  λ={_lambda_binding}")
print(f"  cgt init    : {'✅' if CFG['cgt']['enabled'] else 'off'}  λ_cgt={_lambda_cgt}")
print(f"  landscape   : {'✅' if CFG['landscape']['enabled'] else 'off'}  warmup@{CFG['landscape']['warmup_steps']}")
print(f"  topo        : ✅  warmup@{CFG['topo']['warmup_steps']} (Option A)")
print(f"  L_align     : ✅  λ={CFG['l_align']['lambda_align']}  k={CFG['l_align']['sample_k']}")
print(f"  λ CE/KL/hid : {dist_cfg.alpha} / {dist_cfg.lambda_distill} / {dist_cfg.lambda_hidden}")
print(f"  3-phase CE  : REMOVED")
print(f"  VolCE       : DISABLED")
print(f"  steps       : {trainer.step} → {MAX_STEPS}")

# ══════════════════════════════════════════════════════════════════════════
# ACTIVE INTEGRATION v7
# ══════════════════════════════════════════════════════════════════════════
activate_all_modules_v7(
    trainer            = trainer,
    pcgrad_inst        = pcgrad_inst,
    hpc_guard_v7       = hpc_guard_v7,
    hysteresis_det     = hysteresis_det,
    kuramoto_scheduler = kuramoto_scheduler,
    geo_ctrl           = geo_ctrl,
    extra_loss_hooks   = getattr(trainer, "_extra_loss_hooks", []),
    lambda_align       = CFG["l_align"]["lambda_align"],
    align_sample_k     = CFG["l_align"]["sample_k"],
)

# ── CSV flush ─────────────────────────────────────────────────────────────
import csv as _csv, threading

def _flush(trainer, log_dir):
    # v7: ensure l_align and k_eff always in fieldnames
    V7_EXTRA_FIELDS = ["k_eff", "l_align", "aux_loss"]
    for fname, hist in [("training_metrics.csv", trainer.train_hist),
                        ("val_metrics.csv",      trainer.val_hist)]:
        if not hist: continue
        tmp = log_dir / (fname + ".tmp")
        dst = log_dir / fname
        # Union of all keys across all rows + v7 extras
        all_keys = list(hist[0].keys())
        for field in V7_EXTRA_FIELDS:
            if field not in all_keys:
                all_keys.append(field)
        # Fill missing keys with 0.0
        filled = [{**{k: 0.0 for k in V7_EXTRA_FIELDS}, **row} for row in hist]
        with open(tmp, "w", newline="") as f:
            w = _csv.DictWriter(f, fieldnames=all_keys, extrasaction="ignore")
            w.writeheader(); w.writerows(filled)
        tmp.replace(dst)

_FLUSH_INTERVAL = 300
_flush_stop     = threading.Event()

def _bg_flush_loop():
    while not _flush_stop.wait(_FLUSH_INTERVAL):
        try:
            _flush(trainer, _log_dir)
            print(f"  [auto-flush] ✅ step={getattr(trainer,'step','?')}"
                  f"  train={len(getattr(trainer,'train_hist',[]))}"
                  f"  val={len(getattr(trainer,'val_hist',[]))}  → {_log_dir.name}/")
        except Exception as _fe:
            print(f"  [auto-flush] ⚠️  {_fe}")

_flush_thread = threading.Thread(target=_bg_flush_loop, daemon=True, name="csv-flusher")
_flush_thread.start()
print(f"  [auto-flush] ✅ Thread iniciada — {_FLUSH_INTERVAL//60} min")

# ── Train ─────────────────────────────────────────────────────────────────
remaining = MAX_STEPS - trainer.step
print(f"\n  🚀 Training {remaining:,} steps  (H100 est. ~{remaining/15000/60:.1f}h)")
t0 = time.time()
try:
    train_hist, val_hist = trainer.train(train_loader, val_loader)
except RadiusCollapseAbort as e:
    print(f"  ⚡ RadiusCollapse: {e}")
    train_hist = getattr(trainer, "train_hist", [])
    val_hist   = getattr(trainer, "val_hist",  [])
finally:
    _flush_stop.set()
    _flush(trainer, _log_dir)
    print(f"  [auto-flush] 🛑 flush final")

elapsed = time.time() - t0
print(controller.summary())

try:
    trainer.save(is_best=True)
    torch.save(student.state_dict(), _ckpt_dir / "student_v7.pt")
    print(f"  💾 Saved → {_ckpt_dir}")
except Exception as e:
    print(f"  ⚠️  Save failed: {e}")

rdcs = [float(r["rdc_ema"]) for r in val_hist if r.get("rdc_ema") not in (None,"","nan")]
ppls = [float(r["val_ppl"]) for r in val_hist if r.get("val_ppl")]
rdc_star = round(sum(rdcs[-5:])/min(5,len(rdcs)),2) if rdcs else None
best_ppl = round(min(ppls),1) if ppls else None
print(f"\n  ✅ {elapsed/3600:.1f}h  rdc*={rdc_star}  best_ppl={best_ppl}")
if best_ppl and best_ppl <= 35:
    print("  🏆 SUPEROU GPT-2-small! (PPL ≤ 35)")
elif best_ppl and best_ppl <= 100:
    print("  ✅ Competitivo (PPL ≤ 100)")


Run → /content/drive/MyDrive/HydraPaper_v4/logs/hydra_v4_cgt_binding
  Checkpoint dir: /content/drive/MyDrive/HydraPaper_v4/checkpoints/hydra_v4_cgt_binding
  Exists: True
  Checkpoints: ['distill_v2_ckpt_1000.pt', 'distill_v2_latest.pt', 'distill_v2_best.pt']
  🔄 Resuming — distill_v2_best.pt
  ℹ️  Running in eager mode (torch.compile disabled)
🔄 Loading checkpoint: /content/drive/MyDrive/HydraPaper_v4/checkpoints/hydra_v4_cgt_binding/distill_v2_best.pt


/tmp/ipykernel_1848/967305218.py:94: UserWarning: [Paper2] OTED active: Origin-Tangent Euclidean Distillation.
  trainer = DistillationTrainerV2(


✅ Loaded v2 checkpoint from step 1000
   ✅ Resumed at step=1000  best_val=inf
  ✅ Resumed at step 1000  best_val=inf
  ✅ Post-load override: deg_eq_action=none, λ_kl=0.01, λ_hid=0.05, T=1.0
  [AdaptiveController] Initialised — mode=single-GPU  L1@500  L2@every_eval  L3@200
  [AdaptiveController] ✅ Activo  (L1@500  rad_max=4.0)
  [v3] ✅ Binding loss hook: λ=0.02  threshold=Γ≥0.3
  [v4] ✅ CGT coherence hook: λ=0.01 (F1 Lorentz regularization)
  [Gemini] ✅ HPCGuard + HysteresisDetector active
  [Gemini] ✅ VolumeWeightedCE active (blend=0.1)
  [Gemini] ✅ PCGrad active
  [Gemini] ✅ GeometricController active
  [SGDR] Starting fresh from step 1000 — no restart needed
  [Magnetic] ✅ FrictionAwareCouplingScheduler active
  [Magnetic] ✅ MagneticFrictionExtension active
  lm_head  : SublatticeLMHead
  binding  : H-AKORN  λ=0.02  Γ_target=0.3
  cgt init : ✅ enabled  λ_cgt=0.01
  λ CE/KL/hid : 1.0 / 0.01 / 0.05
  steps    : 1000 → 200000
  controller: L1@500  L2@eval  L3@200
  [auto-flush] Thread 

Cell 8 — Chat with trained model

In [ ]:
import torch
import torch.nn.functional as F

MAX_NEW_TOKENS = 150
TEMPERATURE    = 0.7
TOP_K          = 50
TOP_P          = 0.9
REPETITION_PEN = 1.3

@torch.no_grad()
def generate(model, tokenizer, prompt, max_new_tokens=MAX_NEW_TOKENS,
             temperature=TEMPERATURE, top_k=TOP_K, top_p=TOP_P,
             rep_penalty=REPETITION_PEN):
    model.eval()
    ids = tokenizer.encode(prompt, return_tensors="pt").to(DEVICE)
    gen = ids.clone()
    for _ in range(max_new_tokens):
        ctx    = gen[:, -CFG["model"]["n_positions"]:]
        logits = model(ctx)["logits"][:, -1, :]
        if temperature != 1.0: logits = logits / max(temperature, 1e-6)
        if rep_penalty != 1.0:
            for t in set(gen[0].tolist()):
                logits[0,t] = logits[0,t]/rep_penalty if logits[0,t]>0 else logits[0,t]*rep_penalty
        if top_k > 0:
            v, _ = torch.topk(logits, min(top_k, logits.size(-1)))
            logits[logits < v[:, [-1]]] = float("-inf")
        if top_p < 1.0:
            s_logits, s_idx = torch.sort(logits, descending=True)
            cum = s_logits.softmax(-1).cumsum(-1)
            s_logits[cum - s_logits.softmax(-1) > top_p] = float("-inf")
            logits = torch.zeros_like(logits).scatter_(1, s_idx, s_logits)
        next_tok = torch.multinomial(F.softmax(logits, -1), 1)
        gen = torch.cat([gen, next_tok], dim=1)
        if next_tok.item() == tokenizer.eos_token_id: break
    return tokenizer.decode(gen[0, ids.shape[1]:], skip_special_tokens=True)

# ── Load best checkpoint ──────────────────────────────────────────────────
_ckpt = _ckpt_dir / "student_competitive.pt"
if _ckpt.exists():
    student.load_state_dict(torch.load(str(_ckpt), map_location=DEVICE))
    student.eval()
    print(f"✅ Loaded {_ckpt.name}")

print("╔══════════════════════════════════════════════════════╗")
print("║         HyDRA-Competitive Chat  (H⁵¹², K=1)        ║")
print("║   64M params · OpenWebText · AngularLMHeadV2        ║")
print("╚══════════════════════════════════════════════════════╝")
print("Commands: /temp /topk /topp /tokens /rep /quit\n")

gen_cfg = dict(temperature=TEMPERATURE, top_k=TOP_K, top_p=TOP_P,
               max_new_tokens=MAX_NEW_TOKENS, rep_penalty=REPETITION_PEN)
context = ""
while True:
    try: user = input("You: ").strip()
    except (EOFError, KeyboardInterrupt): print("\n[end]"); break
    if not user: continue
    if user.startswith("/"):
        p = user.split()
        try:
            if p[0]=="/quit": break
            elif p[0]=="/temp":   gen_cfg["temperature"]    = float(p[1])
            elif p[0]=="/topk":   gen_cfg["top_k"]          = int(p[1])
            elif p[0]=="/topp":   gen_cfg["top_p"]          = float(p[1])
            elif p[0]=="/tokens": gen_cfg["max_new_tokens"] = int(p[1])
            elif p[0]=="/rep":    gen_cfg["rep_penalty"]    = float(p[1])
            print(f"  {p[0][1:]} = {p[1] if len(p)>1 else "?"}")
        except: print(f"  usage: {p[0]} <value>")
        continue
    resp = generate(student, tokenizer, context + user + " ", **gen_cfg)
    print(f"HyDRA: {resp}\n")
    context = (context + user + " " + resp + " ")[-400:]